> 在线讲义：[第 10 章 · KV 怎么变小 ②：窗口、稀疏、线性与混合注意力](https://wikieden.github.io/ai-infra-from-constraints/lessons/10-kv-sparse-linear.html) · 本书《从约束推导 AI Infra》

# 第 10 章 · KV 怎么变小 ②：窗口、稀疏、线性与混合注意力

配套讲义：仓库里 `ai-infra/notes/lessons/10-kv-sparse-linear.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 V4-Flash 8K | 逐层加 | `v4_flash_state(8192)` | 保存 59.125 MiB，读 27.625 MiB |
| 第二步 小例子 | 外积累加 | `linear_attention([...])` | $\mathcal{S}_2 = [[2,3],[1,1]]$，$o = [2,3]$ |
| 第二步 Kimi K3 KDA | $69 \times 96 \times 128^2 \times 4$ | `recurrent_mib(69, 96, 128)` | 414 MiB，读写 828 MiB |
| 第三步 交点 | $M / c$ | `h_star(61.875 * 2**20, 20 * 1024)` | 3168 |
| 第四步 V4.1 / V4 | 两个式子 | `v41_bytes_per_token()`、`v4_bytes_per_token()` | 890、3514.25 |

DeepSeek V4-Flash / V4.1 Flash、Kimi K3、Qwen3.6 的结构与字节数取自 AII 第 2 章 §2.3.3–§2.3.6；线性注意力的 2 维小例子是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `v4_flash_state`**：V4-Flash（2 窗口层 + 21 CSA + 20 HCA，BF16 参考格式）：(保存 MiB, 最后一次查询读取 MiB)。

In [ ]:
def v4_flash_state(S: int, win: int = 128, d: int = 512, d_idx: int = 128, b: int = 2) -> tuple[float, float]:
    """V4-Flash（2 窗口层 + 21 CSA + 20 HCA，BF16 参考格式）：(保存 MiB, 最后一次查询读取 MiB)。"""
    csa, hca, w = S // 4, S // 128, min(S, win)
    keep = 43 * w * d * b + 21 * csa * (d + d_idx) * b + 20 * hca * d * b
    read = 43 * w * d * b + 21 * (csa * d_idx + min(512, csa) * d) * b + 20 * hca * d * b
    return keep / 2**20, read / 2**20

**函数 `linear_attention`**：S_t = S_{t-1} + k vᵀ，o = qᵀ S。返回 (最终状态, 输出)；状态大小始终 d_k × d_v。

In [ ]:
def linear_attention(tokens: list[tuple[list[float], list[float]]], q: list[float]) -> tuple[list[list[float]], list[float]]:
    """S_t = S_{t-1} + k vᵀ，o = qᵀ S。返回 (最终状态, 输出)；状态大小始终 d_k × d_v。"""
    d_k, d_v = len(tokens[0][0]), len(tokens[0][1])
    state = [[0.0] * d_v for _ in range(d_k)]
    for k, v in tokens:
        for i in range(d_k):
            for j in range(d_v):
                state[i][j] += k[i] * v[j]  # 外积累加
    out = [sum(q[i] * state[i][j] for i in range(d_k)) for j in range(d_v)]
    return state, out

**函数 `recurrent_mib`**：递推状态（MiB）= 层数 × 头数 × d × d × b，不随上下文变化。

In [ ]:
def recurrent_mib(layers: int, heads: int, d: int, b: int = 4) -> float:
    """递推状态（MiB）= 层数 × 头数 × d × d × b，不随上下文变化。"""
    return layers * heads * d * d * b / 2**20

**函数 `h_star`**：混合模型固定状态与逐 token 部分相等的上下文长度 H* = M_fixed / c_state。

In [ ]:
def h_star(fixed_bytes: float, per_token_bytes: float) -> float:
    """混合模型固定状态与逐 token 部分相等的上下文长度 H* = M_fixed / c_state。"""
    return fixed_bytes / per_token_bytes

**函数 `v41_bytes_per_token`**：V4.1 Flash：3 层每 2 token 一条 + 1 层每 token 一条，每条主记录 + 索引。

In [ ]:
def v41_bytes_per_token(main: int = 288, idx: int = 68) -> float:
    """V4.1 Flash：3 层每 2 token 一条 + 1 层每 token 一条，每条主记录 + 索引。"""
    return (3 / 2 + 1) * (main + idx)

**函数 `v4_bytes_per_token`**：V4-Flash：21 层 4:1（主 + 索引）+ 20 层 128:1（只有主记录）。

In [ ]:
def v4_bytes_per_token(main: int = 584, idx: int = 68) -> float:
    """V4-Flash：21 层 4:1（主 + 索引）+ 20 层 128:1（只有主记录）。"""
    return 21 * (main + idx) / 4 + 20 * main / 128

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：V4-Flash，8192 个 token，BF16 参考格式")
keep, read = v4_flash_state(8192)
print(f"  保存 {keep:.3f} MiB，最后一次查询读 {read:.3f} MiB（Qwen3-8B 同样 8K 各 1152 MiB）")

In [ ]:
print("\n第二步：线性注意力小例子（编的）")
state, out = linear_attention([([1, 0], [2, 3]), ([0, 1], [1, 1])], [1, 0])
print(f"  S_2 = {state}，q = [1, 0] 时 o = {out}")
kda = recurrent_mib(69, 96, 128)
print(f"  Kimi K3 的 KDA：{kda:.0f} MiB，每步读写 {2 * kda:.0f} MiB")

In [ ]:
print("\n第三步：混合模型的交点")
print(f"  Qwen3.6：61.875 MiB ÷ 20 KiB = {h_star(61.875 * 2**20, 20 * 1024):.0f}")
print(f"  Kimi K3：433.406 MiB ÷ 27 KiB ≈ {h_star(433.406 * 2**20, 27 * 1024):.0f}")

In [ ]:
print("\n第四步：每 token 的全局 KV 增量")
print(f"  V4.1 Flash {v41_bytes_per_token():.0f} B，V4-Flash {v4_bytes_per_token():.2f} B，相差 {v4_bytes_per_token() / v41_bytes_per_token():.2f} 倍")

## 3. 代码里才看得到的

1. `v4_flash_state` 只算 BF16 参考格式；表里 V4-Flash 的 27.455 / 12.556 MiB 是生产混合格式（FP8/BF16 主记录 584 B、MXFP4 索引），数值接近但不是同一个量。
2. 压缩器的 FP32 缓冲（约 11.6 MiB）没算进 `keep`，和原文 59.125 MiB 的口径一致。
3. `linear_attention` 的状态在循环里只有一个 2×2 列表，不管输入多长都不变大，这就是"固定状态"。

## 4. 自己改一改

- 用 `v4_flash_state` 扫 $S$ = 1024、8192、65536，看保存和读取的比值怎么变。
- 给 `linear_attention` 喂 1000 个随机 token，确认状态还是 2×2。
- 把 `h_star` 的每 token 增量减半，看交点往哪边移。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 近的看窗口，远的看小结，挑之前先翻目录；
> 摘要一张卡，读写都定额；
> 月租加流量，交点 H\* 定谁说了算；
> 存的少不等于读的少，三个数一起看。